In [ ]:
from pymongo import MongoClient
from pymongo.operations import InsertOne
from datetime import datetime
import time

def copy_specific_date_data(
    mongodb_uri='mongodb://localhost:27017/',
    database_name='QUANTAXIS',
    source_collection='stock_min',
    target_collection='stock_min_cn',
    target_date='2026-08-08',
    batch_size=10000
):
    """复制指定日期的数据到新集合"""
    
    # 连接数据库
    client = MongoClient(mongodb_uri)
    db = client[database_name]
    
    source = db[source_collection]
    target = db[target_collection]
    
    print(f"开始复制 {target_date} 的数据...")
    start_time = time.time()
    
    # 1. 查询指定日期的数据（根据你的数据结构，date字段是String类型）
    query = {"date": target_date}
    
    # 如果date字段是Date类型，使用下面这行
    # query = {"date": datetime(2026, 8, 8)}
    
    # 统计符合条件的文档数
    total_docs = source.count_documents(query)
    print(f"找到 {total_docs} 条 {target_date} 的数据")
    
    if total_docs == 0:
        print("⚠️ 没有找到数据，请检查日期格式是否正确")
        print(f"查询条件: {query}")
        return False
    
    # 2. 清空目标集合（可选）
    target.drop()
    print("已清空目标集合")
    
    # 3. 分批复制数据
    operations = []
    copied = 0
    
    # 使用游标分批获取
    cursor = source.find(query)
    
    for doc in cursor:
        operations.append(InsertOne(doc))
        
        if len(operations) >= batch_size:
            target.bulk_write(operations)
            copied += len(operations)
            print(f"进度: {copied}/{total_docs} ({copied/total_docs*100:.2f}%)")
            operations = []
    
    # 处理剩余数据
    if operations:
        target.bulk_write(operations)
        copied += len(operations)
        print(f"进度: {copied}/{total_docs} (100%)")
    
    # 4. 复制索引
    print("\n开始复制索引...")
    indexes = source.index_information()
    
    for index_name, index_info in indexes.items():
        if index_name == '_id_':
            continue
        
        keys = index_info['key']
        target.create_index(
            keys,
            name=index_name,
            unique=index_info.get('unique', False),
            sparse=index_info.get('sparse', False)
        )
        print(f"已创建索引: {index_name}")
    
    elapsed = time.time() - start_time
    print(f"\n✅ 复制完成！")
    print(f"共复制 {copied} 条数据")
    print(f"耗时: {elapsed:.2f} 秒")
    
    return True

# 执行复制
if __name__ == "__main__":
    copy_specific_date_data(
        mongodb_uri='mongodb://localhost:27017/',
        database_name='stock_db',  # 修改为你的数据库名
        source_collection='stock_min',
        target_collection='stock_min_cn',
        target_date='2026-08-08'
    )

In [1]:
from pymongo import MongoClient
from pymongo.operations import InsertOne
from datetime import datetime
import time


In [ ]:
mongodb_uri='mongodb://localhost:27017/'
database_name='quantaxis'
source_collection='stock_min'
target_collection='stock_min_cn'
target_date='2026-08-08'
batch_size=10000

"""复制指定日期的数据到新集合"""

# 连接数据库
client = MongoClient(mongodb_uri)
db = client[database_name]

source = db[source_collection]
target = db[target_collection]

print(f"开始复制 {target_date} 的数据...")
start_time = time.time()

# 1. 查询指定日期的数据（根据你的数据结构，date字段是String类型）
query = {"date": target_date}

# 如果date字段是Date类型，使用下面这行
query = {"date": datetime(2026, 8, 8)}

# 统计符合条件的文档数
total_docs = source.count_documents(query)
# print(f"找到 {total_docs} 条 {target_date} 的数据")

# if total_docs == 0:
#     print("⚠️ 没有找到数据，请检查日期格式是否正确")
#     print(f"查询条件: {query}")
#     return False

# # 2. 清空目标集合（可选）
# target.drop()
# print("已清空目标集合")

# # 3. 分批复制数据
# operations = []
# copied = 0

# # 使用游标分批获取
# cursor = source.find(query)

# for doc in cursor:
#     operations.append(InsertOne(doc))

#     if len(operations) >= batch_size:
#         target.bulk_write(operations)
#         copied += len(operations)
#         print(f"进度: {copied}/{total_docs} ({copied/total_docs*100:.2f}%)")
#         operations = []

# # 处理剩余数据
# if operations:
#     target.bulk_write(operations)
#     copied += len(operations)
#     print(f"进度: {copied}/{total_docs} (100%)")

# # 4. 复制索引
# print("\n开始复制索引...")
# indexes = source.index_information()

# for index_name, index_info in indexes.items():
#     if index_name == '_id_':
#         continue

#     keys = index_info['key']
#     target.create_index(
#         keys,
#         name=index_name,
#         unique=index_info.get('unique', False),
#         sparse=index_info.get('sparse', False)
#     )
#     print(f"已创建索引: {index_name}")

# elapsed = time.time() - start_time
# print(f"\n✅ 复制完成！")
# print(f"共复制 {copied} 条数据")
# print(f"耗时: {elapsed:.2f} 秒")

# return True

开始复制 2026-08-08 的数据...


In [ ]:
from pymongo import MongoClient
from pymongo.operations import InsertOne
import time

def copy_data_without_count(
    mongodb_uri='mongodb://localhost:27017/',
    database_name='stock_db',
    source_collection='stock_min',
    target_collection='stock_min_cn',
    target_date='2026-08-08',
    batch_size=10000
):
    """不提前计数，直接复制"""
    
    client = MongoClient(mongodb_uri)
    db = client[database_name]
    
    source = db[source_collection]
    target = db[target_collection]
    
    print(f"开始复制 {target_date} 的数据...")
    start_time = time.time()
    
    # 清空目标集合
    target.drop()
    print("已清空目标集合")
    
    # 直接查询并复制
    query = {"date": target_date}
    cursor = source.find(query).batch_size(batch_size)  # 设置批量大小
    
    operations = []
    copied = 0
    
    for doc in cursor:
        operations.append(InsertOne(doc))
        
        if len(operations) >= batch_size:
            target.bulk_write(operations)
            copied += len(operations)
            print(f"已复制: {copied} 条")
            operations = []
    
    if operations:
        target.bulk_write(operations)
        copied += len(operations)
        print(f"已复制: {copied} 条")
    
    elapsed = time.time() - start_time
    print(f"\n✅ 复制完成！共复制 {copied} 条数据")
    print(f"耗时: {elapsed:.2f} 秒")
    
    return copied

# 执行
copy_data_without_count(
    database_name='stock_db',
    target_date='2026-08-08'
)

In [2]:
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['stock_db']

query = {"date": "2026-08-08"}

# 分析查询执行计划
explain_result = db['stock_min'].find(query).explain()

# 查看是否使用了索引
if 'winningPlan' in explain_result:
    plan = explain_result['winningPlan']
    if 'indexName' in plan.get('inputStage', {}):
        print(f"使用了索引: {plan['inputStage']['indexName']}")
    else:
        print("⚠️ 没有使用索引，建议创建索引")
else:
    print(explain_result)

{'queryPlanner': {'plannerVersion': 1, 'namespace': 'stock_db.stock_min', 'indexFilterSet': False, 'parsedQuery': {'date': {'$eq': '2026-08-08'}}, 'winningPlan': {'stage': 'EOF'}, 'rejectedPlans': []}, 'executionStats': {'executionSuccess': True, 'nReturned': 0, 'executionTimeMillis': 3, 'totalKeysExamined': 0, 'totalDocsExamined': 0, 'executionStages': {'stage': 'EOF', 'nReturned': 0, 'executionTimeMillisEstimate': 0, 'works': 1, 'advanced': 0, 'needTime': 0, 'needYield': 0, 'saveState': 0, 'restoreState': 0, 'isEOF': 1}, 'allPlansExecution': []}, 'serverInfo': {'host': 'DESKTOP-R0MPONM', 'port': 27017, 'version': '4.2.6', 'gitVersion': '20364840b8f1af16917e4c23c1b5f5efd8b352f8'}, 'ok': 1.0}


In [6]:
from pymongo import MongoClient
from pymongo.operations import InsertOne
import time

def fast_copy_data(
    mongodb_uri='mongodb://localhost:27017/',
    database_name='quantaxis',
    source_collection='stock_min',
    target_collection='stock_min_cn',
    target_date='2026-08-07',
    batch_size=10000
):
    """
    快速复制数据（优化版）
    不使用 count_documents()，直接复制
    """
    
    client = MongoClient(mongodb_uri)
    db = client[database_name]
    
    source = db[source_collection]
    target = db[target_collection]
    
    print(f"开始复制 {target_date} 的数据...")
    print("提示: 如果数据量大，请耐心等待")
    start_time = time.time()
    
    # 1. 检查并创建索引（如果不存在）
    existing_indexes = [idx['name'] for idx in source.list_indexes()]
    if 'date_index' not in existing_indexes:
        print("创建 date 索引以加速查询...")
        source.create_index("date")
    
    # 2. 清空目标集合
    target.drop()
    print("已清空目标集合")
    
    # 3. 使用 cursor 分批复制
    query = {"date": target_date}
    cursor = source.find(query).batch_size(batch_size)
    
    operations = []
    copied = 0
    
    for doc in cursor:
        operations.append(InsertOne(doc))
        
        if len(operations) >= batch_size:
            target.bulk_write(operations)
            copied += len(operations)
            print(f"已复制: {copied} 条")
            operations = []
    
    if operations:
        target.bulk_write(operations)
        copied += len(operations)
        print(f"已复制: {copied} 条")
    
    # 4. 复制索引
    print("\n复制索引...")
    indexes = source.index_information()
    for index_name, index_info in indexes.items():
        if index_name == '_id_':
            continue
        
        keys = index_info['key']
        target.create_index(
            keys,
            name=index_name,
            unique=index_info.get('unique', False),
            sparse=index_info.get('sparse', False)
        )
        print(f"  已创建索引: {index_name}")
    
    elapsed = time.time() - start_time
    print(f"\n✅ 复制完成！")
    print(f"共复制 {copied} 条数据")
    print(f"耗时: {elapsed:.2f} 秒")
    
    return copied

# 执行
if __name__ == "__main__":
    fast_copy_data(
        database_name='quantaxis',  # 修改为你的数据库名
        target_date='2026-08-07'
    )

开始复制 2026-08-07 的数据...
提示: 如果数据量大，请耐心等待
已清空目标集合

复制索引...
  已创建索引: date_1

✅ 复制完成！
共复制 0 条数据
耗时: 0.15 秒


In [4]:
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['stock_db']

# 一行代码搞定，速度最快
db['stock_min'].aggregate([
    {"$match": {"date": "2026-08-07"}},
    {"$out": "stock_min_cn"}
])

print(f"复制完成！共 {db['stock_min_cn'].count_documents({})} 条数据")

复制完成！共 0 条数据


In [ ]:
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['stock_db']  # 修改为你的数据库名

# 使用 $merge 替代 $out
pipeline = [
    {"$match": {"date": "2026-08-07"}},  # 先测试 2026-08-07
    {"$merge": {
        "into": "stock_min_cn",
        "whenMatched": "replace",
        "whenNotMatched": "insert"
    }}
]

# 执行并确保游标被消费
result = list(db['stock_min'].aggregate(pipeline))

count = db['stock_min_cn'].count_documents({})
print(f"✅ 复制完成！共 {count} 条数据")